In [ ]:
from mp_api.client import MPRester
from pathlib import Path
import json
from datetime import datetime

api_key = ""

PATH = Path.cwd() / "materials_project_reference_data"
PATH.mkdir(parents=True, exist_ok=True)
LOG_FILE = PATH / "mp_pull_log.json"

material = "Gamma-In2Se3" 
material_id = "mp-612740"

# Load the existing log (or start a new one)
if LOG_FILE.exists():
    with open(LOG_FILE) as f:
        pull_log = json.load(f)
else:
    pull_log = {}

# Add / update this material's entry
pull_log[material_id] = {
    "material": material,
    "last_pulled": datetime.now().isoformat(timespec="seconds"),
}

with open(LOG_FILE, "w") as f:
    json.dump(pull_log, f, indent=4)



In [48]:
import json
from pathlib import Path

try:
    with MPRester(api_key) as mpr:
        phonon_doc = mpr.materials.phonon.search(material_id)

        # In addition, you can specfiy the phonon calculation methodology by using `phonon_method`,
        # the default is `None` so all results for mp-19907 are returned. Other options: "dfpt" or "pheasy"
        phonon_doc = mpr.materials.phonon.search(
            material_ids=[material_id], phonon_method="pheasy"
        )
except Exception as e:
    print(f"Error fetching phonon data: {e}")
    
print(phonon_doc)

Output()

Output()

✓ Retrieved 1 SummaryDoc documents in 0.6s

Error fetching phonon data: No phonon data found for material ID(s) ['mp-612740'] with phonon_method='pheasy'.
[]


In [49]:
# Bandstructures
import json

# Set the file save directories
SAVE_PATH = Path.cwd() / "materials_project_reference_data" / "band_structure"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}-phonondata.json"

# Execute fetch phonon data
try:
    with MPRester(api_key) as mpr:
        # Bandstructure path convention can be specified by using `path_type`. Default is "setyawan_curtarolo".
        phonon_bs = mpr.materials.phonon.get_bandstructure_from_material_id(
        material_id,
            phonon_method="pheasy",
            path_type="latimer_munro",
        )
        if not phonon_bs:
             raise ValueError(f"No phonon data found for material ID: {material_id}")

        phonon_dict = phonon_bs.to_pmg.as_dict()


        with open(output_fname, "w") as json_file:
            json.dump(phonon_dict, json_file, indent=4)
        print(f"Phonon data successfully saved to {output_fname}")

except Exception as e:
    print(f"Error fetching phonon data: {e}")
    


Output()

Error fetching phonon data: No phonon data found for material ID mp-612740.


In [50]:
# Force Constants (currently only available for `phonon_method="pheasy"`)
import numpy as np
from phonopy.file_IO import write_FORCE_CONSTANTS

# Set the file save directories
SAVE_PATH = Path.cwd() / "materials_project_reference_data" / "force_constants"
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# Set the output file name and save location
output_fname = SAVE_PATH / f"{material}_force_constants"

# Execute fetch phonon daya
try:
    with MPRester(api_key) as mpr:
        phonon_force_constants = mpr.materials.phonon.get_forceconstants_from_material_id(
            material_id, phonon_method="pheasy"
        )
    fc = np.array(phonon_force_constants)
    print(fc.shape)
    write_FORCE_CONSTANTS(fc, filename=str(SAVE_PATH / output_fname))  
except Exception as e:
    print(f"Error fetching force constant data {e}")



Output()

Error fetching force constant data No phonon data found for material ID mp-612740.


In [51]:
# DOS
# with MPRester(api_key) as mpr:
#    phonon_dos = mpr.materials.phonon.get_dos_from_material_id(
#        material_id, phonon_method="pheasy"
#    )
